# 02 — Feature Engineering

In general we do want our data to be a little more detailed so that we can better predict outcomes once we reach the modeling and selection phase of this project. Currently we have very basic team stats like current record, points scored, and other key team statistics. However using these features we can actually make some of our own that can add more information to our analysis

For example, it is pretty common knowledge that veiwing a teams current win streak could be an indicator of the result of the current matchup. For example we would be inclined to say that if a team is currently on a 3 game win streak, they are more likely to win the next game, thus extending the streak. the inverse would also be true, a team on a 3 game losing streak would be seen as less likely to win the next game. We will explore this later in the eda portion of this project but for now we can add it. 

Additionally a teams record in their previous 10 games could also be a good indicator of winning chance for current matchup. A team who has performed well in their last 10 would be more favored to win, and vice versa. 

We also look to add rest days column to the data, this could be useful as the more time a team has to rest, the better their players will perform. Further analysis will help us determine the importance of rest days

Lastly we aim to add rolling averages for each team individually. As the season progresses it would make sense that good performing teams will have higher rolling averages and thus would be favored when entering a new matchup.

For all matchups we also calculate the opposing teams rolling averages and add them to the data to better forecast matchups

Loads all raw team game log CSVs, applies every feature function from `src/features.py`, and saves the result to `data/processed/features.csv`.

In [ ]:
import sys
from pathlib import Path


def _find_project_root() -> Path:
    """Locate the project root (the dir containing data/raw) regardless of
    where the Jupyter kernel was started from."""
    # 1. Walk up from the current working directory.
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "raw").is_dir():
            return candidate
    # 2. Fallback to the known absolute location of this project.
    fallback = Path("/Users/yourrem/PROJECTS/DATA_SCIENCE/Time_Series/sports_analytics")
    if (fallback / "data" / "raw").is_dir():
        return fallback
    raise FileNotFoundError(
        "Could not locate project root containing data/raw. "
        f"Searched from cwd={Path.cwd()}"
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.features import (
    add_home_away_flag,
    add_last_10_wins,
    add_opponent_features,
    add_rest_days,
    add_rolling_averages,
    add_season_record,
    add_win_streak,
)

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
print(f"PROJECT_ROOT = {PROJECT_ROOT}")

# stat columns to compute rolling averages for
STAT_COLS = ["PTS", "FG_PCT", "FG3_PCT", "FT_PCT", "REB", "AST", "TOV", "STL", "BLK"]

## 1. Load Raw Team Game Logs

In [ ]:
csv_paths = sorted(RAW_DIR.glob("team_game_logs_*.csv"))
print(f"{len(csv_paths)} team-season files found")

raw_df = pd.concat([pd.read_csv(p) for p in csv_paths], ignore_index=True)
print(f"Combined shape: {raw_df.shape}")
raw_df.head(3)

## 2. Apply Features Per Team

Each feature function sorts by date internally, so we loop over individual team-season files rather than applying across the whole DataFrame at once. This keeps win streaks, rest days, and rolling averages scoped to each team's own game history.

In [ ]:
enriched = []

for p in csv_paths:
    df = pd.read_csv(p)

    df = add_home_away_flag(df)
    df = add_rest_days(df)
    df = add_rolling_averages(df, stat_cols=STAT_COLS, window=5)
    df = add_rolling_averages(df, stat_cols=STAT_COLS, window=10)
    df = add_win_streak(df)
    df = add_last_10_wins(df)
    df = add_season_record(df)

    enriched.append(df)

features_df = pd.concat(enriched, ignore_index=True)
print(f"Feature matrix shape: {features_df.shape}")
features_df.head(3)

In [ ]:
features_df = add_opponent_features(features_df, stat_cols=STAT_COLS)
print(f"Shape after opponent features: {features_df.shape}")

## 3. Encode Target Variable

In [ ]:
features_df["target"] = (features_df["WL"] == "W").astype(int)

print("Class balance:")
print(features_df["target"].value_counts(normalize=True).rename({1: "Win", 0: "Loss"}))

## 4. Inspect New Columns

In [ ]:
feature_cols = (
    ["is_home", "rest_days", "win_streak", "last_10_w", "season_wins", "season_losses"]
    + [f"{c}_roll5"  for c in STAT_COLS]
    + [f"{c}_roll10" for c in STAT_COLS]
    + ["opp_rest_days", "opp_win_streak", "opp_last_10_w"]
    + [f"{c}_diff_roll5" for c in STAT_COLS]
)

print(f"{len(feature_cols)} feature columns:")
print(feature_cols)

features_df[["GAME_DATE", "MATCHUP", "WL"] + feature_cols + ["target"]].head(10)

In [ ]:
null_counts = features_df[feature_cols].isnull().sum()
print("Null counts per feature column:")
print(null_counts[null_counts > 0])

## 5. Save to Processed

In [ ]:
out_path = PROCESSED_DIR / "features.csv"
features_df.to_csv(out_path, index=False)
print(f"Saved {len(features_df):,} rows × {len(features_df.columns)} columns → {out_path}")